# neuro — Standalone Stock/Option Forecaster (no GitHub needed)

All the code is written inline by this notebook. Just **Runtime → Run all**.
Works on a Chromebook: everything runs in your browser on a free Colab GPU.

⚠️ Educational tool only. Markets are noisy and unpredictable — not financial advice.

## 1. Install dependencies

In [ ]:
!pip install -q torch numpy pandas yfinance scikit-learn matplotlib pyyaml

## 2. Write the project files

In [ ]:
import os
os.makedirs('src', exist_ok=True)
print('ready')

In [ ]:
%%writefile src/__init__.py
"""A small, trainable neural network for forecasting stock / option prices."""

__version__ = "0.1.0"


In [ ]:
%%writefile src/config.py
"""Configuration loading and the typed config dataclasses.

Values come from ``config.yaml`` by default and can be overridden by keyword
arguments (typically parsed from the command line in ``cli.py``).
"""

from dataclasses import dataclass, field, fields, is_dataclass
from typing import Any, List, Optional

import yaml


@dataclass
class DataConfig:
    ticker: str = "AAPL"
    start: str = "2015-01-01"
    end: Optional[str] = None
    interval: str = "1d"
    csv: Optional[str] = None  # load from a local CSV instead of downloading
    features: List[str] = field(default_factory=lambda: ["Open", "High", "Low", "Close", "Volume"])
    target: str = "Close"
    # How the model predicts the target:
    #   "logreturn" -> predict day-to-day log returns, rebuild price from the
    #                  last actual price (stationary, recommended).
    #   "price"     -> predict the raw price level (prone to mean-reverting to
    #                  the historical average; only for short/stationary series).
    target_mode: str = "logreturn"


@dataclass
class WindowConfig:
    lookback: int = 60
    horizon: int = 5


@dataclass
class ModelConfig:
    hidden_size: int = 128
    num_layers: int = 2
    dropout: float = 0.2
    bidirectional: bool = False


@dataclass
class TrainConfig:
    epochs: int = 60
    batch_size: int = 64
    lr: float = 1e-3
    weight_decay: float = 1e-5
    val_split: float = 0.15
    patience: int = 10
    seed: int = 42
    device: str = "auto"
    resume: bool = False  # continue from existing checkpoint instead of fresh init


@dataclass
class PathsConfig:
    checkpoint: str = "checkpoints/model.pt"
    output_dir: str = "outputs"


@dataclass
class Config:
    data: DataConfig = field(default_factory=DataConfig)
    window: WindowConfig = field(default_factory=WindowConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    train: TrainConfig = field(default_factory=TrainConfig)
    paths: PathsConfig = field(default_factory=PathsConfig)


def _build(cls, data: dict) -> Any:
    """Recursively build a (possibly nested) dataclass from a plain dict."""
    if not is_dataclass(cls):
        return data
    kwargs = {}
    for f in fields(cls):
        if data and f.name in data:
            value = data[f.name]
            if is_dataclass(f.type) or (isinstance(f.type, type) and is_dataclass(f.type)):
                kwargs[f.name] = _build(f.type, value)
            else:
                kwargs[f.name] = value
    return cls(**kwargs)


def load_config(path: str = "config.yaml") -> Config:
    """Load configuration from a YAML file, falling back to defaults."""
    try:
        with open(path, "r") as fh:
            raw = yaml.safe_load(fh) or {}
    except FileNotFoundError:
        raw = {}
    return _build(Config, raw)


def apply_overrides(cfg: Config, overrides: dict) -> Config:
    """Apply a flat dict of overrides (e.g. {"ticker": "MSFT", "epochs": 10}).

    Each key is matched against the fields of every sub-config; ``None`` values
    are ignored so unset CLI flags don't clobber file/default values.
    """
    sections = [cfg.data, cfg.window, cfg.model, cfg.train, cfg.paths]
    for key, value in overrides.items():
        if value is None:
            continue
        for section in sections:
            if hasattr(section, key):
                setattr(section, key, value)
                break
    return cfg


In [ ]:
%%writefile src/data.py
"""Data download and feature engineering.

Prices are fetched from Yahoo Finance via ``yfinance``. On top of the raw OHLCV
columns we add a few classic technical-analysis features that help a model
forecast short-term moves. Everything here returns a tidy ``pandas.DataFrame``
indexed by date.
"""

from __future__ import annotations

from typing import List, Optional

import numpy as np
import pandas as pd


def download_prices(
    ticker: str,
    start: str = "2015-01-01",
    end: Optional[str] = None,
    interval: str = "1d",
) -> pd.DataFrame:
    """Download OHLCV data for ``ticker`` from Yahoo Finance.

    Returns a DataFrame with columns Open, High, Low, Close, Volume indexed by
    date. Raises a clear error if nothing comes back (bad symbol / no network).
    """
    import yfinance as yf

    df = yf.download(
        ticker,
        start=start,
        end=end,
        interval=interval,
        auto_adjust=True,
        progress=False,
    )
    if df is None or df.empty:
        raise ValueError(
            f"No data returned for ticker '{ticker}'. Check the symbol, the "
            f"date range, or your network connection."
        )

    # yfinance can return a MultiIndex (column, ticker) when given one symbol.
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.rename(columns=str.title)
    keep = ["Open", "High", "Low", "Close", "Volume"]
    df = df[[c for c in keep if c in df.columns]].copy()
    df.index.name = "Date"
    return df.dropna()


def add_technical_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add a handful of momentum / volatility indicators.

    The new columns are appended; rows with NaNs introduced by rolling windows
    are dropped at the end so the result is model-ready.
    """
    out = df.copy()
    close = out["Close"]

    # Returns
    out["Return"] = close.pct_change()
    out["LogReturn"] = np.log(close).diff()

    # Moving averages and their ratio to price
    out["SMA_10"] = close.rolling(10).mean()
    out["SMA_30"] = close.rolling(30).mean()
    out["SMA_ratio"] = out["SMA_10"] / out["SMA_30"]

    # Exponential moving averages -> MACD
    ema_12 = close.ewm(span=12, adjust=False).mean()
    ema_26 = close.ewm(span=26, adjust=False).mean()
    out["MACD"] = ema_12 - ema_26
    out["MACD_signal"] = out["MACD"].ewm(span=9, adjust=False).mean()

    # Rolling volatility of returns
    out["Volatility_10"] = out["Return"].rolling(10).std()

    # Relative Strength Index (RSI, 14-day)
    out["RSI_14"] = _rsi(close, period=14)

    return out.dropna()


def _rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))


def load_csv(path: str) -> pd.DataFrame:
    """Load OHLCV data from a local CSV (offline / bring-your-own-data path).

    The CSV must have a date column (named Date, or the first column) and the
    usual OHLCV columns (case-insensitive). Useful when there's no network or
    you have your own historical data / option chain.
    """
    df = pd.read_csv(path)
    date_col = "Date" if "Date" in df.columns else df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col).sort_index()
    df.index.name = "Date"
    df.columns = [str(c).title() for c in df.columns]
    return df.dropna()


TARGET_COL = "__target__"


def apply_target_mode(df: pd.DataFrame, price_col: str, mode: str):
    """Return (df, target_col) prepared for the requested target mode.

    - "price": predict the raw price level directly (target_col == price_col).
    - "logreturn": add a ``__target__`` column of daily log returns of the
      price; the model predicts those and price is rebuilt at inference from the
      last actual price. This keeps the target stationary and anchored to the
      present, avoiding drift toward the multi-year average price.
    """
    df = df.copy()
    if mode == "price":
        return df, price_col
    if mode == "logreturn":
        df[TARGET_COL] = np.log(df[price_col]).diff()
        df = df.dropna()
        return df, TARGET_COL
    raise ValueError(f"Unknown target_mode '{mode}'. Use 'price' or 'logreturn'.")


def build_dataset(
    ticker: str,
    start: str,
    end: Optional[str],
    interval: str,
    features: List[str],
    target: str,
    with_technicals: bool = True,
    csv: Optional[str] = None,
) -> pd.DataFrame:
    """End-to-end: load (download or CSV), engineer features, select columns.

    The returned frame always contains every requested feature column plus the
    target column. If ``csv`` is given, data is read from that file instead of
    being downloaded from Yahoo Finance.
    """
    df = load_csv(csv) if csv else download_prices(ticker, start, end, interval)
    raw_cols = set(df.columns)
    if with_technicals:
        df = add_technical_features(df)

    # Keep the requested features + target, plus any engineered technical
    # indicators (everything that wasn't in the raw OHLCV frame) so the model
    # actually gets to use them as inputs.
    technical_cols = [c for c in df.columns if c not in raw_cols]
    cols = list(dict.fromkeys([*features, *technical_cols, target]))  # de-dupe, keep order

    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(
            f"Requested columns not available: {missing}. "
            f"Available columns: {list(df.columns)}"
        )
    return df[cols].copy()


In [ ]:
%%writefile src/dataset.py
"""Windowing, scaling, and PyTorch ``Dataset`` for sequence forecasting.

The model sees ``lookback`` days of (scaled) features and predicts the next
``horizon`` days of the (scaled) target. Scalers are fit on the training split
only, to avoid look-ahead leakage into validation.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset


@dataclass
class Scalers:
    """Holds the fitted feature/target scalers and column bookkeeping."""

    feature_scaler: StandardScaler
    target_scaler: StandardScaler
    feature_cols: List[str]
    target_col: str

    def transform_features(self, x: np.ndarray) -> np.ndarray:
        return self.feature_scaler.transform(x)

    def inverse_target(self, y: np.ndarray) -> np.ndarray:
        y = np.asarray(y).reshape(-1, 1)
        return self.target_scaler.inverse_transform(y).ravel()


def make_windows(
    features: np.ndarray, target: np.ndarray, lookback: int, horizon: int
) -> Tuple[np.ndarray, np.ndarray]:
    """Slice arrays into (X, y) sliding windows.

    X has shape (n, lookback, n_features); y has shape (n, horizon).
    """
    xs, ys = [], []
    n = len(features)
    for i in range(n - lookback - horizon + 1):
        xs.append(features[i : i + lookback])
        ys.append(target[i + lookback : i + lookback + horizon])
    if not xs:
        raise ValueError(
            f"Not enough rows ({n}) for lookback={lookback} + horizon={horizon}. "
            f"Use a longer history or smaller windows."
        )
    return np.asarray(xs, dtype=np.float32), np.asarray(ys, dtype=np.float32)


def prepare_splits(
    df: pd.DataFrame,
    feature_cols: List[str],
    target_col: str,
    lookback: int,
    horizon: int,
    val_split: float,
) -> Tuple["SequenceDataset", "SequenceDataset", Scalers]:
    """Chronologically split, fit scalers on train, and build datasets.

    The split is done on the raw rows *before* windowing so that no validation
    row ever appears inside a training window.
    """
    n = len(df)
    n_val = int(n * val_split)
    # Guarantee both splits can produce at least one window.
    min_rows = lookback + horizon
    n_val = max(min_rows, min(n_val, n - min_rows))

    train_df = df.iloc[: n - n_val]
    # Carry lookback rows of context into validation so its first window is valid.
    val_df = df.iloc[n - n_val - lookback :]

    scalers = _fit_scalers(train_df, feature_cols, target_col)

    train_ds = SequenceDataset(train_df, scalers, lookback, horizon)
    val_ds = SequenceDataset(val_df, scalers, lookback, horizon)
    return train_ds, val_ds, scalers


def _fit_scalers(df: pd.DataFrame, feature_cols: List[str], target_col: str) -> Scalers:
    feature_scaler = StandardScaler().fit(df[feature_cols].values)
    target_scaler = StandardScaler().fit(df[[target_col]].values)
    return Scalers(feature_scaler, target_scaler, list(feature_cols), target_col)


class SequenceDataset(Dataset):
    """Wraps a DataFrame slice into scaled (X, y) sliding-window tensors."""

    def __init__(
        self,
        df: pd.DataFrame,
        scalers: Scalers,
        lookback: int,
        horizon: int,
    ):
        features = scalers.transform_features(df[scalers.feature_cols].values)
        target = scalers.target_scaler.transform(df[[scalers.target_col]].values).ravel()
        self.X, self.y = make_windows(features, target, lookback, horizon)

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        return torch.from_numpy(self.X[idx]), torch.from_numpy(self.y[idx])


In [ ]:
%%writefile src/model.py
"""The forecasting network: an LSTM encoder with a linear projection head.

The LSTM consumes a sequence of feature vectors and the final hidden state is
projected to ``horizon`` future target values. It's deliberately simple and
fast to train on a laptop while still being a real recurrent net.
"""

from __future__ import annotations

import torch
import torch.nn as nn


class LSTMForecaster(nn.Module):
    def __init__(
        self,
        n_features: int,
        horizon: int,
        hidden_size: int = 128,
        num_layers: int = 2,
        dropout: float = 0.2,
        bidirectional: bool = False,
    ):
        super().__init__()
        self.n_features = n_features
        self.horizon = horizon
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        directions = 2 if bidirectional else 1
        self.head = nn.Sequential(
            nn.Linear(hidden_size * directions, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, horizon),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, lookback, n_features)
        out, _ = self.lstm(x)
        last = out[:, -1, :]  # final timestep representation
        return self.head(last)  # (batch, horizon)

    @property
    def config(self) -> dict:
        """Hyperparameters needed to reconstruct this module from a checkpoint."""
        return {
            "n_features": self.n_features,
            "horizon": self.horizon,
            "hidden_size": self.hidden_size,
            "num_layers": self.num_layers,
            "bidirectional": self.bidirectional,
        }


In [ ]:
%%writefile src/train.py
"""Training loop with validation, early stopping, and checkpointing.

A checkpoint bundles the model weights, its architecture config, and the fitted
scalers + column lists so that ``predict.py`` can reload everything it needs to
forecast on fresh data.
"""

from __future__ import annotations

import os
import pickle
import random
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from .config import Config
from .data import apply_target_mode, build_dataset
from .dataset import prepare_splits
from .model import LSTMForecaster


def resolve_device(choice: str = "auto") -> torch.device:
    if choice == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(choice)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def _run_epoch(model, loader, criterion, device, optimizer=None) -> float:
    train = optimizer is not None
    model.train(train)
    total, count = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.set_grad_enabled(train):
            pred = model(x)
            loss = criterion(pred, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        total += loss.item() * x.size(0)
        count += x.size(0)
    return total / max(count, 1)


def train(cfg: Config) -> dict:
    """Train a model end-to-end and write a checkpoint. Returns a summary dict."""
    set_seed(cfg.train.seed)
    device = resolve_device(cfg.train.device)
    print(f"[train] device={device}")

    if cfg.data.csv:
        print(f"[train] loading data from {cfg.data.csv}")
    else:
        print(f"[train] downloading {cfg.data.ticker} ({cfg.data.start} -> {cfg.data.end or 'today'})")
    df = build_dataset(
        ticker=cfg.data.ticker,
        start=cfg.data.start,
        end=cfg.data.end,
        interval=cfg.data.interval,
        features=cfg.data.features,
        target=cfg.data.target,
        csv=cfg.data.csv,
    )
    # Derive the prediction target (raw price vs. log returns). The price
    # columns stay as input features either way.
    df, target_col = apply_target_mode(df, cfg.data.target, cfg.data.target_mode)
    feature_cols = [c for c in df.columns if c != target_col]
    print(
        f"[train] {len(df)} rows, mode={cfg.data.target_mode}, "
        f"target={target_col}, features={feature_cols}"
    )

    train_ds, val_ds, scalers = prepare_splits(
        df,
        feature_cols=feature_cols,
        target_col=target_col,
        lookback=cfg.window.lookback,
        horizon=cfg.window.horizon,
        val_split=cfg.train.val_split,
    )
    print(f"[train] windows: train={len(train_ds)}, val={len(val_ds)}")

    train_loader = DataLoader(train_ds, batch_size=cfg.train.batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.train.batch_size, shuffle=False)

    model = LSTMForecaster(
        n_features=len(feature_cols),
        horizon=cfg.window.horizon,
        hidden_size=cfg.model.hidden_size,
        num_layers=cfg.model.num_layers,
        dropout=cfg.model.dropout,
        bidirectional=cfg.model.bidirectional,
    ).to(device)

    # Optionally continue training from an existing checkpoint instead of
    # starting from random weights. Architecture must match (same feature count
    # and horizon); scalers are re-fit on the current data.
    if getattr(cfg.train, "resume", False) and os.path.exists(cfg.paths.checkpoint):
        prev = torch.load(cfg.paths.checkpoint, map_location=device, weights_only=False)
        if prev.get("model_config") == model.config:
            model.load_state_dict(prev["model_state"])
            print(f"[train] resumed weights from {cfg.paths.checkpoint}")
        else:
            print(
                "[train] WARNING: existing checkpoint architecture differs "
                "(feature count or horizon changed) -> starting from scratch"
            )

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=max(2, cfg.train.patience // 2)
    )

    best_val = float("inf")
    best_state: Optional[dict] = None
    epochs_no_improve = 0

    for epoch in range(1, cfg.train.epochs + 1):
        tr_loss = _run_epoch(model, train_loader, criterion, device, optimizer)
        val_loss = _run_epoch(model, val_loader, criterion, device, None)
        scheduler.step(val_loss)
        print(f"[train] epoch {epoch:3d}/{cfg.train.epochs}  train={tr_loss:.5f}  val={val_loss:.5f}")

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= cfg.train.patience:
                print(f"[train] early stopping at epoch {epoch} (best val={best_val:.5f})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    summary = save_checkpoint(cfg, model, scalers, feature_cols, best_val)
    print(f"[train] saved checkpoint -> {cfg.paths.checkpoint}  (best val={best_val:.5f})")
    return summary


def save_checkpoint(cfg: Config, model, scalers, feature_cols, best_val: float) -> dict:
    os.makedirs(os.path.dirname(cfg.paths.checkpoint) or ".", exist_ok=True)
    payload = {
        "model_state": model.state_dict(),
        "model_config": model.config,
        "scalers": pickle.dumps(scalers),
        "feature_cols": feature_cols,
        "price_col": cfg.data.target,
        "target_mode": cfg.data.target_mode,
        "val_split": cfg.train.val_split,
        "lookback": cfg.window.lookback,
        "horizon": cfg.window.horizon,
        "data": {
            "ticker": cfg.data.ticker,
            "interval": cfg.data.interval,
            "features": cfg.data.features,
            "csv": cfg.data.csv,
        },
        "best_val": best_val,
    }
    torch.save(payload, cfg.paths.checkpoint)
    return {"checkpoint": cfg.paths.checkpoint, "best_val": best_val}


In [ ]:
%%writefile src/predict.py
"""Load a trained checkpoint and forecast the next ``horizon`` prices.

Also supports a simple back-test plot comparing predictions against the most
recent actual prices.
"""

from __future__ import annotations

import pickle
from typing import List, Optional

import numpy as np
import pandas as pd
import torch

from .data import build_dataset
from .model import LSTMForecaster


def load_checkpoint(path: str, device: Optional[torch.device] = None):
    device = device or torch.device("cpu")
    payload = torch.load(path, map_location=device, weights_only=False)
    model = LSTMForecaster(**payload["model_config"]).to(device)
    model.load_state_dict(payload["model_state"])
    model.eval()
    scalers = pickle.loads(payload["scalers"])
    return model, scalers, payload


def _price_col(payload) -> str:
    # "price_col" is the new key; fall back to the old "target_col" for
    # checkpoints trained before target modes existed.
    return payload.get("price_col", payload.get("target_col", "Close"))


def _latest_frame(payload, start: str = "2015-01-01") -> pd.DataFrame:
    """Re-download recent data using the same settings the model was trained on."""
    data = payload["data"]
    return build_dataset(
        ticker=data["ticker"],
        start=start,
        end=None,
        interval=data["interval"],
        features=data["features"],
        target=_price_col(payload),
        csv=data.get("csv"),
    )


def forecast(checkpoint: str, device: Optional[torch.device] = None) -> pd.DataFrame:
    """Predict the next ``horizon`` target values after the latest available data.

    Returns a DataFrame indexed by future (business-day) dates with the
    predicted price.
    """
    device = device or torch.device("cpu")
    model, scalers, payload = load_checkpoint(checkpoint, device)
    lookback = payload["lookback"]
    feature_cols: List[str] = payload["feature_cols"]

    df = _latest_frame(payload)
    if len(df) < lookback:
        raise ValueError(f"Need at least {lookback} rows; got {len(df)}.")

    window = df[feature_cols].values[-lookback:]
    x = scalers.transform_features(window).astype(np.float32)
    x = torch.from_numpy(x).unsqueeze(0).to(device)  # (1, lookback, n_features)

    with torch.no_grad():
        pred_scaled = model(x).cpu().numpy().ravel()
    raw = scalers.inverse_target(pred_scaled)

    price_col = _price_col(payload)
    if payload.get("target_mode", "price") == "logreturn":
        # raw are predicted log returns; rebuild the price path from the last
        # actual price so the forecast is anchored to today, not the long-run mean.
        last_price = float(df[price_col].iloc[-1])
        preds = last_price * np.exp(np.cumsum(raw))
    else:
        preds = raw

    last_date = df.index[-1]
    future_dates = pd.bdate_range(start=last_date, periods=len(preds) + 1)[1:]
    result = pd.DataFrame({"predicted_" + price_col: preds}, index=future_dates)
    result.index.name = "Date"
    return result


def backtest_plot(checkpoint: str, out_path: str, device: Optional[torch.device] = None) -> str:
    """Plot the last stretch of actual prices plus the forward forecast."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    device = device or torch.device("cpu")
    _, _, payload = load_checkpoint(checkpoint, device)
    df = _latest_frame(payload)
    target = _price_col(payload)
    preds = forecast(checkpoint, device)

    recent = df[target].iloc[-120:]
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.plot(recent.index, recent.values, label=f"Actual {target}", color="#1f77b4")
    pred_col = "predicted_" + target
    ax.plot(preds.index, preds[pred_col].values, "o--", label="Forecast", color="#d62728")
    ax.axvline(df.index[-1], color="gray", linestyle=":", alpha=0.7)
    ax.set_title(f"{payload['data']['ticker']} — {len(preds)}-step forecast")
    ax.set_xlabel("Date")
    ax.set_ylabel("Price")
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(out_path, dpi=120)
    plt.close(fig)
    return out_path


In [ ]:
%%writefile src/cli.py
"""Command-line interface.

Examples
--------
Train on Apple with defaults from config.yaml::

    python -m src.cli train --ticker AAPL --epochs 50

Forecast the next few days from a saved checkpoint::

    python -m src.cli predict --plot

"""

from __future__ import annotations

import argparse
import os

from .config import apply_overrides, load_config
from .train import resolve_device, train


def _add_common(p: argparse.ArgumentParser) -> None:
    p.add_argument("--config", default="config.yaml", help="path to YAML config")
    p.add_argument("--ticker", help="ticker symbol, e.g. AAPL")
    p.add_argument("--checkpoint", help="checkpoint path")


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        prog="stock-forecaster",
        description="Train an LSTM to forecast stock/option prices.",
    )
    sub = parser.add_subparsers(dest="command", required=True)

    # train
    pt = sub.add_parser("train", help="download data and train a model")
    _add_common(pt)
    pt.add_argument("--csv", help="load data from a local CSV instead of downloading")
    pt.add_argument(
        "--target_mode",
        choices=["logreturn", "price"],
        help="predict log returns (recommended) or raw price level",
    )
    pt.add_argument("--start", help="history start date YYYY-MM-DD")
    pt.add_argument("--end", help="history end date YYYY-MM-DD")
    pt.add_argument("--interval", help="bar interval, e.g. 1d, 1h")
    pt.add_argument("--lookback", type=int, help="days of history fed to the model")
    pt.add_argument("--horizon", type=int, help="days to forecast ahead")
    pt.add_argument("--epochs", type=int)
    pt.add_argument("--batch_size", type=int)
    pt.add_argument("--lr", type=float)
    pt.add_argument("--device", choices=["auto", "cpu", "cuda"])
    pt.add_argument(
        "--resume",
        action="store_true",
        default=None,
        help="continue training from the existing checkpoint instead of starting fresh",
    )

    # predict
    pp = sub.add_parser("predict", help="forecast future prices from a checkpoint")
    _add_common(pp)
    pp.add_argument("--plot", action="store_true", help="also save a forecast plot")
    pp.add_argument("--device", choices=["auto", "cpu", "cuda"])

    # backtest
    pb = sub.add_parser("backtest", help="measure held-out accuracy vs a naive baseline")
    _add_common(pb)
    pb.add_argument("--eval_split", type=float, help="fraction of recent data to test on")
    pb.add_argument("--device", choices=["auto", "cpu", "cuda"])

    return parser


def main(argv=None) -> int:
    args = build_parser().parse_args(argv)
    cfg = load_config(args.config)

    overrides = {
        k: v
        for k, v in vars(args).items()
        if k not in {"command", "config", "plot"} and v is not None
    }
    cfg = apply_overrides(cfg, overrides)

    if args.command == "train":
        train(cfg)
        return 0

    if args.command == "predict":
        from .predict import backtest_plot, forecast

        device = resolve_device(cfg.train.device)
        if not os.path.exists(cfg.paths.checkpoint):
            raise SystemExit(
                f"No checkpoint at '{cfg.paths.checkpoint}'. Run `train` first."
            )
        preds = forecast(cfg.paths.checkpoint, device)
        print("\nForecast:")
        print(preds.to_string(float_format=lambda v: f"{v:,.2f}"))
        if args.plot:
            os.makedirs(cfg.paths.output_dir, exist_ok=True)
            out = os.path.join(cfg.paths.output_dir, f"{cfg.data.ticker}_forecast.png")
            backtest_plot(cfg.paths.checkpoint, out, device)
            print(f"\nSaved plot -> {out}")
        return 0

    if args.command == "backtest":
        from .backtest import format_report, run_backtest

        device = resolve_device(cfg.train.device)
        if not os.path.exists(cfg.paths.checkpoint):
            raise SystemExit(
                f"No checkpoint at '{cfg.paths.checkpoint}'. Run `train` first."
            )
        metrics = run_backtest(cfg.paths.checkpoint, device, eval_split=args.eval_split)
        print("\n" + format_report(metrics))
        return 0

    return 1


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile config.yaml
# Configuration for the stock/option price forecaster.
# Edit these values, or override any of them on the command line, e.g.
#   python -m src.cli train --ticker AAPL --epochs 50 --lookback 90

data:
  ticker: AAPL          # symbol to download from Yahoo Finance
  start: "2015-01-01"   # history start date
  end: null             # null = up to today
  interval: "1d"        # 1d, 1h, etc. (intraday history is limited by Yahoo)
  # Columns used as model inputs. The target ("Close") must be present.
  features: ["Open", "High", "Low", "Close", "Volume"]
  target: "Close"
  # "logreturn" predicts daily % changes and rebuilds the price from the last
  # actual price (recommended). "price" predicts the raw level and tends to
  # drift toward the historical average -> looks like a fake crash.
  target_mode: "logreturn"

window:
  lookback: 60          # number of past days fed into the model
  horizon: 5            # number of future days to predict at once

model:
  hidden_size: 128
  num_layers: 2
  dropout: 0.2
  bidirectional: false

train:
  epochs: 60
  batch_size: 64
  lr: 0.001
  weight_decay: 0.00001
  val_split: 0.15       # fraction of the (chronological) tail used for validation
  patience: 10          # early-stopping patience (epochs without val improvement)
  seed: 42
  device: "auto"        # auto | cpu | cuda

paths:
  checkpoint: "checkpoints/model.pt"
  output_dir: "outputs"


## 3. Train
Edit the ticker / epochs / window to taste. Colab has network, so it downloads real data.

In [ ]:
!python -m src.cli train --ticker AAPL --epochs 50 --lookback 60 --horizon 5 --device auto

## 4. Forecast + plot

In [ ]:
!python -m src.cli predict --plot

In [ ]:
from IPython.display import Image
Image('outputs/AAPL_forecast.png')